# StravaGANte Syntetic Data Retriever
The dataset is collected using OpenRouteService, an open source project which exposes free APIs.

In [21]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

profiles = [
    'driving-car',
    'driving-hgv',
    'cycling-regular',
    'cycling-road',
    'cycling-mountain',
    'cycling-electric'
]
output_filedir = 'Data/Syntetic/'
output_filename = 'output.gpx'
os.makedirs(output_filedir, exist_ok=True)

load_dotenv()
ors_token = os.getenv("OpenRouteServiceApiKey")
print("Token ok.") if ors_token else print("Token not found.")

Token ok.


IMDB 250 top films to scrape locations

In [22]:
from googlesearch import search
import csv

latlong_file = '../Data/latlong_movies.csv'
df_movies = pd.read_csv('../Data/IMDB_Top_250_Movies.csv', usecols=[1])
movie_names = df_movies['name'].tolist()

movie_locations = {}

if os.path.isfile(latlong_file):
    existing_data = pd.read_csv(latlong_file)
    existing_movies = existing_data['name'].tolist()
else:
    existing_movies = []
    with open(latlong_file, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['name', 'latlong_url'])

for movie in movie_names:
    query = f"site:latlong.net {movie}"
    search_results = list(search(query, num=1))

    if search_results:
        if movie not in existing_movies or pd.isna(existing_data.loc[existing_data['name'] == movie, 'latlong_url']).any():
            with open(latlong_file, mode='a', newline='') as file:
                writer.writerow([movie, search_results[0]])

        # df_movies['latlong_url'] = df_movies['name'].map(movie_locations)
        # df_movies.to_csv('../Data/IMDB_Top_250_Movies.csv', index=False)

HTTPError: HTTP Error 429: Too Many Requests

POST Request to OpenRouteService

In [ ]:
pz = [11.88713795374955,45.41111616690249]
body = {"coordinates":[pz,[11.931983982915773, 45.42370290352176], pz]} # torre archimede and prato della valle

headers = {
    'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
    'Authorization': ors_token,
    'Content-Type': 'application/json; charset=utf-8'
}
call = requests.post(f'https://api.openrouteservice.org/v2/directions/{profiles[2]}/gpx', json=body, headers=headers)

if call.status_code == 200:
    gpx_file_path = os.path.join(output_filedir, output_filename)
    with open(gpx_file_path, 'w') as file:
        file.write(call.text)